In [1]:
# -*- coding: utf-8 -*-
"""
Caracterização das condições de imagem dos 50 vídeos do dataset.
Responde aos pedidos R3-02 e R3-16 da revisão do SBrT 2026.

Por que rodar localmente: os vídeos somam ~16 GB em C:\\Vi_DataSet_S_K_Frame,
volume inviável de trafegar. O script lê tudo em disco e exporta só as
estatísticas.

Uso:
    python analise_condicoes_video.py

Saída:
    C:/DataSet_SoccerKeyFrame - Org/Condicoes_Video_Dataset.xlsx
      aba "por_video"  -> uma linha por vídeo
      aba "por_split"  -> agregado treino / validação / teste
      aba "amostras"   -> série amostrada, para gráficos

Requer: opencv-python, numpy, pandas, openpyxl (todos já usados no projeto).
"""

import os
import cv2
import numpy as np
import pandas as pd

RAIZ = r"C:\Vi_DataSet_S_K_Frame"
SAIDA = r"C:/DataSet_SoccerKeyFrame - Org/Condicoes_Video_Dataset.xlsx"

# 1 amostra a cada N frames. 15 -> 2 amostras/s, mesma taxa da extração
# de frames dos vídeos com PiP. Aumente para acelerar.
PASSO = 15

# mapeamento pasta -> split, seguindo a organização do repositório
SPLITS = {
    os.path.join(RAIZ, "TRAIN", "LE_C_FOG"): ("treino", "com PiP"),
    os.path.join(RAIZ, "TRAIN", "LE_S_FOG"): ("treino", "sem PiP"),
    os.path.join(RAIZ, "VALID"): ("validação", None),
    os.path.join(RAIZ, "TEST"): ("teste", None),
}


def classe_por_nome(nome):
    return "sem PiP" if nome.upper().startswith("SEM_FOG") else "com PiP"


def analisa(caminho, passo=PASSO):
    """Percorre o vídeo amostrando 1 frame a cada `passo` e devolve as séries."""
    cap = cv2.VideoCapture(caminho)
    if not cap.isOpened():
        return None

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    lum, sat, nitidez, contraste, p05, p95 = [], [], [], [], [], []
    corte_cena = []
    hist_ant = None
    idx = 0

    while True:
        ok = cap.grab()
        if not ok:
            break
        if idx % passo == 0:
            ok, frame = cap.retrieve()
            if not ok:
                break
            # reduz antes de medir: as estatísticas são estáveis e fica bem mais rápido
            pequeno = cv2.resize(frame, (480, 270), interpolation=cv2.INTER_AREA)
            hsv = cv2.cvtColor(pequeno, cv2.COLOR_BGR2HSV)
            cinza = cv2.cvtColor(pequeno, cv2.COLOR_BGR2GRAY)

            lum.append(float(cinza.mean()))
            contraste.append(float(cinza.std()))
            sat.append(float(hsv[:, :, 1].mean()))
            nitidez.append(float(cv2.Laplacian(cinza, cv2.CV_64F).var()))
            p05.append(float(np.percentile(cinza, 5)))
            p95.append(float(np.percentile(cinza, 95)))

            # detecção de corte de cena por correlação de histograma
            h = cv2.calcHist([cinza], [0], None, [64], [0, 256])
            cv2.normalize(h, h)
            if hist_ant is not None:
                corte_cena.append(float(cv2.compareHist(hist_ant, h, cv2.HISTCMP_CORREL)))
            hist_ant = h
        idx += 1

    cap.release()
    return dict(fps=fps, n_frames=idx, lum=lum, sat=sat, nitidez=nitidez,
                contraste=contraste, p05=p05, p95=p95, corte=corte_cena)


def rotula_iluminacao(lum_media, p95_medio):
    """Rótulo grosseiro, só para agrupar. Ajuste os cortes se destoar do que você vê."""
    if lum_media < 80:
        return "escuro / noturno"
    if lum_media < 115:
        return "intermediário"
    return "claro / diurno"


linhas, amostras = [], []

for pasta, (split, classe_fixa) in SPLITS.items():
    if not os.path.isdir(pasta):
        print(f"[aviso] pasta não encontrada, pulando: {pasta}")
        continue
    for nome in sorted(os.listdir(pasta)):
        if not nome.lower().endswith((".mp4", ".avi", ".mov")):
            continue
        caminho = os.path.join(pasta, nome)
        print(f"analisando [{split}] {nome} ...", flush=True)
        r = analisa(caminho)
        if r is None or not r["lum"]:
            print(f"  [erro] não foi possível ler {nome}")
            continue

        classe = classe_fixa or classe_por_nome(nome)
        lum = np.array(r["lum"])
        # um corte de cena é uma correlação de histograma abaixo de 0,7
        n_cortes = int(np.sum(np.array(r["corte"]) < 0.70)) if r["corte"] else 0

        linhas.append({
            "video": nome,
            "split": split,
            "classe": classe,
            "n_frames": r["n_frames"],
            "fps": round(r["fps"], 3),
            "amostras": len(lum),
            "luminancia_media": round(float(lum.mean()), 2),
            "luminancia_dp": round(float(lum.std()), 2),
            "luminancia_min": round(float(lum.min()), 2),
            "luminancia_max": round(float(lum.max()), 2),
            "p05_medio": round(float(np.mean(r["p05"])), 2),
            "p95_medio": round(float(np.mean(r["p95"])), 2),
            "contraste_medio": round(float(np.mean(r["contraste"])), 2),
            "saturacao_media": round(float(np.mean(r["sat"])), 2),
            "nitidez_media": round(float(np.mean(r["nitidez"])), 1),
            "nitidez_dp": round(float(np.std(r["nitidez"])), 1),
            "cortes_de_cena": n_cortes,
            "iluminacao": rotula_iluminacao(lum.mean(), np.mean(r["p95"])),
        })

        for i, v in enumerate(lum):
            amostras.append({
                "video": nome, "split": split, "classe": classe,
                "amostra": i, "frame": i * PASSO,
                "luminancia": round(float(v), 2),
                "contraste": round(float(r["contraste"][i]), 2),
                "nitidez": round(float(r["nitidez"][i]), 1),
            })

df = pd.DataFrame(linhas)
df_am = pd.DataFrame(amostras)

por_split = (df.groupby(["split", "classe"])
               .agg(videos=("video", "count"),
                    luminancia_media=("luminancia_media", "mean"),
                    luminancia_dp_entre_videos=("luminancia_media", "std"),
                    contraste_medio=("contraste_medio", "mean"),
                    saturacao_media=("saturacao_media", "mean"),
                    nitidez_media=("nitidez_media", "mean"),
                    cortes_por_video=("cortes_de_cena", "mean"))
               .round(2).reset_index())

os.makedirs(os.path.dirname(SAIDA), exist_ok=True)
with pd.ExcelWriter(SAIDA, engine="openpyxl") as w:
    df.sort_values(["split", "classe", "video"]).to_excel(w, "por_video", index=False)
    por_split.to_excel(w, "por_split", index=False)
    df_am.to_excel(w, "amostras", index=False)

print(f"\nPronto: {SAIDA}")
print(f"{len(df)} vídeos analisados, {len(df_am)} amostras.\n")
print(por_split.to_string(index=False))
print("\nDistribuição de iluminação por split:")
print(pd.crosstab(df["split"], df["iluminacao"]).to_string())

analisando [treino] CFOG01T_00-28-18.mp4 ...
analisando [treino] CFOG03T_01-26-01.mp4 ...
analisando [treino] CFOG05T_00-18-04.mp4 ...
analisando [treino] CFOG06T_01-49-28.mp4 ...
analisando [treino] CFOG08T_00-40-00.mp4 ...
analisando [treino] CFOG10T_00-53-13.mp4 ...
analisando [treino] CFOG11T_01-12-11.mp4 ...
analisando [treino] CFOG13T_01-00-16.mp4 ...
analisando [treino] CFOG15T_01-35-00.mp4 ...
analisando [treino] CFOG16T_01-39-15.mp4 ...
analisando [treino] CFOG18T_01-18-07.mp4 ...
analisando [treino] CFOG20T_00-15-00.mp4 ...
analisando [treino] CFOG21T_00-20-13.mp4 ...
analisando [treino] CFOG23T_00-45-16.mp4 ...
analisando [treino] CFOG25T_01-05-14.mp4 ...
analisando [treino] CFOG26T_01-15-07.mp4 ...
analisando [treino] CFOG28T_01-30-16.mp4 ...
analisando [treino] CFOG30T_01-40-09.mp4 ...
analisando [treino] SEM_FOG_01.mp4 ...
analisando [treino] SEM_FOG_03.mp4 ...
analisando [treino] SEM_FOG_05.mp4 ...
analisando [treino] SEM_FOG_06.mp4 ...
analisando [treino] SEM_FOG_08.mp4

C:\Users\thiag\AppData\Local\Temp\ipykernel_12720\3626341523.py:167: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.sort_values(["split", "classe", "video"]).to_excel(w, "por_video", index=False)
C:\Users\thiag\AppData\Local\Temp\ipykernel_12720\3626341523.py:168: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  por_split.to_excel(w, "por_split", index=False)
C:\Users\thiag\AppData\Local\Temp\ipykernel_12720\3626341523.py:169: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df_am.to_excel(w, "amostras", index=False)



Pronto: C:/DataSet_SoccerKeyFrame - Org/Condicoes_Video_Dataset.xlsx
50 vídeos analisados, 12014 amostras.

    split  classe  videos  luminancia_media  luminancia_dp_entre_videos  contraste_medio  saturacao_media  nitidez_media  cortes_por_video
    teste com PiP       6            109.66                       22.19            36.43           119.83        1470.27             11.67
    teste sem PiP       4            113.72                        5.64            44.18           127.65        1638.68             14.00
   treino com PiP      18            110.60                       21.16            38.18           112.83        1534.89             14.44
   treino sem PiP      12            111.36                       17.70            35.82           109.25        1464.13             13.58
validação com PiP       6            117.20                       30.59            38.94           112.01        1328.10             16.67
validação sem PiP       4             92.20              